## RQ4: What are the most occurring ingredients in NOVA 3 and NOVA 4 that lead to health problems?

The correlation between health risks and higher NOVA 3 (processed) and 
NOVA 4 (ultra-processed) consumption can develop potential health risk(s) 
as shown in Q3. This research question further explores the root cause of 
these illnesses among Americans where ingredients tell the truth — utilizing 
visualizations to show the occurring NOVA 3 and NOVA 4 ingredients found 
in different food categories that lead to health problems.

In [1]:
import pandas as pd
from collections import Counter

df = pd.read_csv("branded_food_cleaned_classified.csv")
print(f"Loaded: {len(df):,} rows")

Loaded: 1,963,676 rows


In [2]:
df_nova3 = df[df["nova_category"] == "NOVA 3 - Processed"].copy()
df_nova4 = df[df["nova_category"] == "NOVA 4 - Ultra-Processed"].copy()

print(f"NOVA 3 products: {len(df_nova3):,}")
print(f"NOVA 4 products: {len(df_nova4):,}")

NOVA 3 products: 325,482
NOVA 4 products: 1,076,861


In [3]:
def clean_ingredient(ing):
    ing = ing.strip().upper()
    ing = ing.rstrip(")].,:;")
    ing = ing.lstrip("(")
    return ing.strip()

def get_all_ingredients(dataframe, nova_label):
    ingredient_product_count = Counter()
    
    for item in dataframe["ingredients"].dropna():
        parts = set([clean_ingredient(ing) for ing in item.split(",")])
        parts = {p for p in parts if p and len(p) > 1}
        ingredient_product_count.update(parts)
    
    result = pd.DataFrame(ingredient_product_count.most_common(), 
                          columns=["ingredient", "product_count"])
    result["% of products"] = (result["product_count"] / len(dataframe) * 100).round(2)
    
    print(f"Total unique ingredients in {nova_label}: {len(result):,}")
    return result

nova3_all = get_all_ingredients(df_nova3, "NOVA 3")
nova4_all = get_all_ingredients(df_nova4, "NOVA 4")

Total unique ingredients in NOVA 3: 50,714
Total unique ingredients in NOVA 4: 264,332


### Most Occurring Ingredients

Before filtering, these are the top 50 most occurring ingredients in 
NOVA 3 and NOVA 4 products. At first glance, the lists appear harmless - 
dominated by salt, sugar, and water. These are ingredients consumers 
recognize and expect.

In [4]:
# Top 50 most occurring ingredients in NOVA 3
nova3_all.head(50)

,ingredient,product_count,% of products
0,SALT,175601,53.95
1,WATER,148055,45.49
2,SUGAR,82562,25.37
3,ENZYMES,73953,22.72
4,CITRIC ACID,46164,14.18
5,YEAST,44612,13.71
6,FOLIC ACID,41345,12.70
7,NIACIN,38441,11.81
8,NATURAL FLAVOR,36551,11.23
9,REDUCED IRON,32696,10.05


In [5]:
# Top 50 most occurring ingredients in NOVA 4
nova4_all.head(50)

,ingredient,product_count,% of products
0,SALT,617004,57.30
1,SUGAR,535908,49.77
2,WATER,342702,31.82
3,CITRIC ACID,275433,25.58
4,FOLIC ACID,207793,19.30
5,CORN SYRUP,206790,19.20
6,NATURAL FLAVOR,197010,18.29
7,SOY LECITHIN,187311,17.39
8,NIACIN,184603,17.14
9,RIBOFLAVIN,167253,15.53


### Removing Common Ingredients - Revealing the Chemicals

The top 50 above shows what consumers expect to see - salt, sugar, water, 
vitamins, and dairy. But these familiar ingredients mask the chemical 
additives that truly define processed and ultra-processed foods.

By filtering out common everyday ingredients, the hidden additives are 
revealed. These are the ingredients most consumers never read on the 
label - and the ones directly linked to the health risks identified in Q3.

In [6]:
common_ingredients = [
    "salt", "water", "sugar", "sea salt", "cane sugar",
    "niacin", "riboflavin", "folic acid", "reduced iron", 
    "thiamine mononitrate", "thiamin mononitrate",
    "vitamin a palmitate", "vitamin d3", "vitamin b6",
    "enzymes", "yeast", "spices", "spice",
    "milk", "cream", "pasteurized milk", "skim milk", "nonfat milk",
    "wheat flour", "enriched wheat flour", "butter", "whole wheat flour",
    "eggs", "whey", "cheese culture", "cheese cultures",
    "garlic", "garlic powder", "onion", "onion powder",
    "black pepper", "vinegar", "baking soda", "corn starch",
    "malted barley flour", "iron", "wheat gluten", "cornstarch",
    "ascorbic acid", "calcium sulfate", "sodium bicarbonate",
    "yeast extract", "cocoa butter", "nonfat milk"
]

nova3_additives = nova3_all[~nova3_all["ingredient"].str.lower().isin(common_ingredients)]
nova4_additives = nova4_all[~nova4_all["ingredient"].str.lower().isin(common_ingredients)]

In [7]:
print("NOVA 3 - Top 20 Chemical Additives (common ingredients removed)")
nova3_additives.head(20)

NOVA 3 - Top 20 Chemical Additives (common ingredients removed)


,ingredient,product_count,% of products
4,CITRIC ACID,46164,14.18
8,NATURAL FLAVOR,36551,11.23
16,SOYBEAN OIL,27355,8.40
18,NATURAL FLAVORS,24549,7.54
19,SOY LECITHIN,23252,7.14
21,CARRAGEENAN,21454,6.59
22,GUAR GUM,21304,6.55
24,PECTIN,18961,5.83
27,XANTHAN GUM,16559,5.09
31,MONOGLYCERIDES,15213,4.67


In [8]:
print("NOVA 4 - Top 20 Chemical Additives (common ingredients removed)")
nova4_additives.head(20)

NOVA 4 - Top 20 Chemical Additives (common ingredients removed)


,ingredient,product_count,% of products
3,CITRIC ACID,275433,25.58
5,CORN SYRUP,206790,19.20
6,NATURAL FLAVOR,197010,18.29
7,SOY LECITHIN,187311,17.39
10,DEXTROSE,163603,15.19
14,NATURAL FLAVORS,120415,11.18
15,XANTHAN GUM,119465,11.09
17,SOYBEAN OIL,106487,9.89
18,MALTODEXTRIN,104668,9.72
23,GUAR GUM,89216,8.28


In [ ]:
# Saving all ingredients NOVA 3 and NOVA 4

nova3_all.to_csv("NOVA3_all_ingredients.csv", index=False)
nova4_all.to_csv("NOVA4_all_ingredients.csv", index=False)

# Chemical additives only (common ingredients removed)
nova3_additives.to_csv("NOVA3_chemical_additives.csv", index=False)
nova4_additives.to_csv("NOVA4_chemical_additives.csv", index=False)

print(f"Saved NOVA 3 all ingredients: {len(nova3_all):,}")
print(f"Saved NOVA 4 all ingredients: {len(nova4_all):,}")
print(f"Saved NOVA 3 chemical additives: {len(nova3_additives):,}")
print(f"Saved NOVA 4 chemical additives: {len(nova4_additives):,}")

### Chemical Additives - NOVA 3 vs NOVA 4 (Top 20)

After removing common everyday ingredients, the top 20 chemical 
additives reveal a clear escalation from NOVA 3 to NOVA 4:

**NOVA 3 additives** are primarily emulsifiers, thickeners, and 
preservatives — chemicals that modify texture and extend shelf life:
- Citric Acid (14.18%), Carrageenan (6.59%), Guar Gum (6.55%), 
  Pectin (5.83%), Xanthan Gum (5.09%)
- The most concerning entry is Calcium Propionate (3.03%), a 
  preservative linked to irritability and sleep disturbances 
  in children.

**NOVA 4 additives** introduce entirely new categories absent 
from NOVA 3:

**Engineered Sweeteners:**
- Corn Syrup (19.20%), Dextrose (15.19%), Maltodextrin (9.72%), 
  HFCS (8.08%)
- Linked to: Type 2 Diabetes, Obesity, Insulin Resistance, 
  Fatty Liver Disease

**Artificial Colors:**
- Caramel Color (8.04%), Red 40 (6.64%), Blue 1 (5.96%), 
  Yellow 5 (5.79%)
- Linked to: Hyperactivity and attention deficits in children, 
  potential carcinogenic compounds, allergic reactions
- **None of these appear in NOVA 3's top 20**

**Industrial Oils:**
- Palm Oil (8.16%), Canola Oil (6.47%)
- Linked to: Elevated LDL cholesterol, cardiovascular disease, 
  chronic inflammation

**Engineered Flavoring:**
- Natural Flavor (18.29%), Natural and Artificial Flavors (5.92%)
- Designed to override satiety signals and promote overconsumption

> The critical finding: NOVA 3 adds chemicals to **preserve** food. 
> NOVA 4 adds chemicals to **engineer** food - maximizing taste, 
> color, and addictive appeal at the expense of human health.